# Services and the language model

A program can ask the robot computer to start what it needs: a simulated robot, or
the language model. This module lists what it can ask for, starts and stops a
service, handles a refusal, asks the model a question, and tests a program that acts
on the answer.

Run it on the mock first. The mock answers with the same rules and starts nothing,
and has no language model unless you give it one (part 5). In the lab, run it with a
simulated robot: the language model is refused while the real robot is running.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME) if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. What there is

`r.services.status()` lists every service and needs no session. The `tier` says who
can start it: **on request** is your program, **launcher only** is the operator, and
**always on** needs nobody.

In [ ]:
for s in r.services.status():
    print(f"{s.name:11s} {s.state:18s} {s.tier:14s} {s.what}")

## 2. Start and stop

`start()` holds a lease for your program, and g1lab renews it every minute while
your program runs. Ten minutes after the last program using a service stops, or calls
`stop()`, the robot computer stops it, so running your program again a minute later
finds it still up.

`wait` is how many seconds to wait for the service to come up. In the lab, `"sim"`
gives you a simulated robot with no operator needed, then `r.control()` works as
usual. It takes tens of seconds, and it is refused while the real robot is on.

In [ ]:
try:
    s = r.services.start("llm", wait=120)
    print(s.name, s.state, "| programs using it:", s.programs, "| lease left:", s.lease_left_s, "s")
    if s.note:
        print(s.note)
    print("after stop:", r.services.stop("llm").state)
except g1lab.ServiceRefused as e:
    print(e)        # part 3 says why

## 3. When it says no

`g1lab.ServiceRefused` carries the reason as its message. Services that move the
robot start only from the launcher, and some are not built yet. The language model
is also refused while the real robot is running, or when the graphics card is too
full, and the reason gives the numbers.

In [ ]:
for name in ("deploy", "vision"):
    try:
        r.services.start(name)
    except g1lab.ServiceRefused as e:
        print(f"{name}: {e}")

## 4. Ask the model

`r.ask()` returns the model's text. The model runs on the robot computer, so this
works from a laptop, and the first question starts the model if it is not running,
which takes up to a minute. It only writes text: it cannot move the robot, and it
knows nothing about the robot, your program or your last question, so put everything
it needs in `prompt`, and the instruction it should keep to in `system`.

The answer stops at `max_tokens` (default 512, most 2048; a word is one or two
tokens). `answer.truncated` is `True` when it was cut off, and `answer.seconds` is
how long it took.

In [ ]:
try:
    answer = r.ask("Name one thing a robot arm can pick up with a pinch grip.", max_tokens=40)
    print(answer)
    print("cut off:", answer.truncated, "| took", round(answer.seconds, 1), "s")
except g1lab.ServiceRefused as e:
    print(e)        # the graphics card is too full, or the real robot is running
except g1lab.G1LabError as e:
    print(e)        # no answer in time, or the question was refused, with the reason

## 5. Act on the answer

The model can answer something other than what you asked for, so check the reply
before you act on it. To test that without a model, give the mock a function from
prompt to text: `g1lab.mock(name=NAME, llm=lambda prompt: "wave")`. Here the fake model
says `wave` to a greeting and rambles otherwise, so both paths run. In the lab, this
cell uses the real model, on a simulated robot.

In [ ]:
def fake_model(prompt):
    return "Wave" if "hello" in prompt else "I think the robot should probably do something nice."

robot = g1lab.mock(name=NAME, llm=fake_model) if HOST == "mock" else r

def respond(c, heard):
    reply = robot.ask(f"The user said: {heard!r}. Answer with one word: wave or stay.",
                      system="You choose one action. Reply with that word and nothing else.",
                      max_tokens=5)
    choice = reply.strip().strip(".").lower()
    print(f"heard {heard!r} -> {reply!r}")
    if choice == "wave":
        arm = robot.info().home["right_arm"].copy()
        arm[0], arm[3] = -1.2, 0.3           # shoulder_pitch: forward and up; elbow
        for yaw in (0.0, 0.4, -0.4, 0.0):    # shoulder_yaw: the wave
            arm[2] = yaw
            c.arms.move_to(right=arm, duration=0.8)
        c.arms.home()
        print("  waved")
    elif choice == "stay":
        print("  staying put")
    else:
        print("  not an action I know: staying put")

with robot.control() as c:
    respond(c, "hello robot")
    respond(c, "what time is it?")

A question takes seconds, and longer behind other people's: they go one at a time, in
order. A program moving the arms keeps them moving from another thread, so a question
holds up only the line of your program that asked it.

**Try this.** Add a third action, a pose of your own, and change `system` and the
checks to match. Give the fake model an answer that is nearly right, such as `"wave!"`,
and see what your check does with it.

In [ ]:
if robot is not r:
    robot.close()
r.close()